# Phase B Training and Visualization

This notebook is a control panel for the scan-geometry experiment. It uses the project scripts and APIs rather than duplicating training logic.

Typical order:

1. Check the Python/PyTorch/Mamba environment.
2. Build Phase B splits and the run matrix.
3. Select one run for a sanity check or training.
4. Aggregate metrics after runs finish.
5. Generate visualizations.

Before running real Mamba jobs on SSH/H100, run `python scripts/ensure_mamba_ssm.py` in the same environment.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG = ROOT / "configs" / "experiments" / "phase_b_10pct.yaml"
PHASE_A_CONFIG = ROOT / "configs" / "datasets" / "phase_a_candidates.yaml"
PHASE_B_DIR = ROOT / "results" / "phase_b"
VIS_DIR = ROOT / "results" / "visualizations"

def run_cmd(args, check=True):
    args = [str(item) for item in args]
    print("$", " ".join(args))
    return subprocess.run(args, cwd=ROOT, text=True, check=check)

print("ROOT:", ROOT)
print("CONFIG exists:", CONFIG.exists())

## 1. Environment Check

This confirms whether the current notebook kernel can run PyTorch and real `mamba_ssm` jobs.

In [ ]:
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_version:", getattr(torch.version, "cuda", None))
except Exception as exc:
    print("torch import failed:", type(exc).__name__, exc)

try:
    from mamba_ssm import Mamba
    print("mamba_ssm: available")
except Exception as exc:
    print("mamba_ssm: unavailable", type(exc).__name__, exc)

In [ ]:
# Optional on SSH/H100 only. This can compile/install CUDA packages.
# run_cmd([sys.executable, "scripts/ensure_mamba_ssm.py"])

## 2. Inspect Phase B Protocol

The config locks datasets, split policy, 10% low-label protocol, conditions, model shape, and training defaults.

In [ ]:
from scan_geometry.phase_b import PhaseBConfig

cfg = PhaseBConfig.from_yaml(CONFIG)
print("datasets:", [dataset.name for dataset in cfg.datasets])
print("conditions:", [condition.name for condition in cfg.conditions])
print("seeds:", cfg.seeds)
print("output_dir:", cfg.output_dir)
print("model:")
print(json.dumps(cfg.model, indent=2))

## 3. Build Splits and Run Matrix

Run this after all five manifest files exist. If manifests are missing, this cell will fail, which is expected.

In [ ]:
manifest_paths = [ROOT / dataset.manifest for dataset in cfg.datasets]
missing_manifests = [path for path in manifest_paths if not path.exists()]
missing_manifests

In [ ]:
# Build locked 70/10/20 splits and the 10% labeled subset.
if missing_manifests:
    print("Missing manifests. Build them before running splits:")
    for path in missing_manifests:
        print(" -", path)
else:
    run_cmd([sys.executable, "scripts/make_splits.py", "--config", CONFIG])

In [ ]:
# Generate the 90-run matrix.
run_cmd([sys.executable, "scripts/train_matrix.py", "--config", CONFIG])
runs_csv = ROOT / "results" / "phase_b" / "runs.csv"
runs = pd.read_csv(runs_csv)
runs.head(12)

In [ ]:
# Protocol sanity checks: within each dataset, Mamba runs should share model_config_hash and differ by scan_order_hash.
mamba = runs[runs["condition_family"] == "mamba"]
hash_check = mamba.groupby("dataset").agg(
    n_model_hashes=("model_config_hash", "nunique"),
    n_scan_hashes=("scan_order_hash", "nunique"),
    n_runs=("run_id", "count"),
)
hash_check

## 4. Train or Dry-Run One Selected Run

Start with `DRY_RUN = True` to verify metadata and model construction. For local macOS without `mamba_ssm`, use `ALLOW_MAMBA_FALLBACK = True` only for shape/smoke tests. Real Mamba experiments on H100 should set it to `False`.

In [ ]:
DATASET = "CAMUS"
CONDITION = "CNN"  # CNN, Raster-H, Raster-V, Hilbert, LocalWindow, RandomPermute
SEED = 2026
DRY_RUN = True
ALLOW_MAMBA_FALLBACK = False

# Useful for quick smoke tests. Use config defaults for real runs.
OVERRIDE_EPOCHS = None
OVERRIDE_BATCH_SIZE = None
DEVICE = None  # None means config default; use "cpu" or "cuda:0" if needed.

selected = runs[(runs["dataset"] == DATASET) & (runs["condition"] == CONDITION) & (runs["seed"] == SEED)]
assert len(selected) == 1, selected
row = selected.iloc[0].to_dict()
row

In [ ]:
cmd = [
    sys.executable,
    "scripts/train_adapter.py",
    "--config", CONFIG,
    "--dataset", row["dataset"],
    "--condition", row["condition"],
    "--scan", "" if pd.isna(row["scan"]) else row["scan"],
    "--seed", int(row["seed"]),
    "--split-file", row["split_file"],
    "--output-dir", row["output_dir"],
]
if DRY_RUN:
    cmd.append("--dry-run")
if ALLOW_MAMBA_FALLBACK:
    cmd.append("--allow-mamba-fallback")
if OVERRIDE_EPOCHS is not None:
    cmd += ["--epochs", OVERRIDE_EPOCHS]
if OVERRIDE_BATCH_SIZE is not None:
    cmd += ["--batch-size", OVERRIDE_BATCH_SIZE]
if DEVICE is not None:
    cmd += ["--device", DEVICE]

run_cmd(cmd)

### Direct Training Module API

The cell above uses the command-line adapter. The cell below shows the actual PyTorch training module used underneath: `TrainingConfig` and `run_training`. Keep `RUN_DIRECT_MODULE = False` unless you intentionally want to launch training from inside the notebook.

In [ ]:
from scan_geometry.models import ModelConfig, build_phase_b_model, count_parameters
from scan_geometry.phase_b import TrainingConfig, run_training
from scan_geometry.phase_b.model_protocol import normalized_model_protocol

RUN_DIRECT_MODULE = False
DIRECT_EPOCHS = 1
DIRECT_BATCH_SIZE = 2

dataset_obj = next(dataset for dataset in cfg.datasets if dataset.name == row["dataset"])
condition_obj = next(condition for condition in cfg.conditions if condition.name == row["condition"])
protocol = normalized_model_protocol(cfg)

model_payload = dict(protocol)
if ALLOW_MAMBA_FALLBACK:
    model_payload["allow_mamba_fallback"] = True

model_config = ModelConfig.from_payload(
    model_payload,
    input_channels=dataset_obj.input_channels,
    num_classes=dataset_obj.num_classes,
)
model = build_phase_b_model(
    condition_family=condition_obj.family,
    scan_order=condition_obj.scan,
    config=model_config,
)
print(count_parameters(model))

if RUN_DIRECT_MODULE:
    outputs = run_training(
        model=model,
        split_file=row["split_file"],
        output_dir=row["output_dir"],
        dataset_name=row["dataset"],
        condition_name=row["condition"],
        seed=int(row["seed"]),
        input_channels=model_config.input_channels,
        num_classes=model_config.num_classes,
        config=TrainingConfig(
            epochs=DIRECT_EPOCHS,
            batch_size=DIRECT_BATCH_SIZE,
            target_size=int(protocol["input_size"][0]),
            device=DEVICE or "auto",
        ),
    )
    print(outputs)
else:
    print("Direct module training is disabled. Set RUN_DIRECT_MODULE = True to launch it.")

## 5. Batch Command Export

Use this to write one shell command per planned run. On a cluster, you can submit these commands through your scheduler.

In [ ]:
commands_path = ROOT / "results" / "phase_b" / "train_commands.sh"
run_cmd([
    sys.executable,
    "scripts/train_matrix.py",
    "--config", CONFIG,
    "--commands-output", commands_path,
])
print(commands_path)
print(commands_path.read_text(encoding="utf-8").splitlines()[:8])

## 6. Aggregate Metrics

After one or more runs produce `metrics.csv`, aggregate them into `metrics_summary.csv`.

In [ ]:
run_cmd([
    sys.executable,
    "scripts/evaluate_runs.py",
    "--runs-csv", ROOT / "results" / "phase_b" / "runs.csv",
    "--output", ROOT / "results" / "phase_b" / "metrics_summary.csv",
], check=False)

metrics_summary = ROOT / "results" / "phase_b" / "metrics_summary.csv"
if metrics_summary.exists():
    display(pd.read_csv(metrics_summary).head(20))
else:
    print("No metrics_summary.csv yet. Train at least one run first.")

## 7. Visualizations

This is safe before training finishes. Missing source data are recorded in `visualization_manifest.json`.

In [ ]:
case_metrics = ROOT / "results" / "phase_b" / "case_metrics.csv"
cmd = [sys.executable, "scripts/plot_visualizations.py", "--output-dir", VIS_DIR]
if case_metrics.exists():
    cmd += ["--case-metrics", case_metrics]
run_cmd(cmd)

manifest_path = VIS_DIR / "visualization_manifest.json"
if manifest_path.exists():
    print(manifest_path.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in [
    "scan_profile_geometry.png",
    "descriptor_correlation.png",
    "performance_heatmap_dice.png",
    "scan_ranking_by_dataset.png",
    "seed_stability.png",
    "p5_matching_association.png",
]:
    path = VIS_DIR / figure
    if path.exists():
        print(path)
        display(Image(filename=str(path)))